# H2O AutoML 기법의 GPU 네이티브 적용 방향

> RAPIDS LAB 세미나 — H2O의 기법을 커널 재작성 없이 GPU 라이브러리로 적용하는 실용적 방향 분석

---

**핵심 전제**

H2O AutoML은 Java 기반이고 CPU에 종속되어 있어서 코드를 직접 가져다 쓸 수는 없다.  
하지만 H2O가 설계한 **기법과 전략**은 기존 GPU 라이브러리(cuDF, cuML, XGBoost GPU)를 조합하면  
커널 수준의 재작성 없이도 적용할 수 있고, GPU 환경에서 오히려 더 강력해진다.

이 노트북에서는 H2O의 각 기법이 **어떤 GPU 라이브러리의 어떤 API로 대체 가능한지**를 구체적으로 분석한다.

---

## 1. Data Layer: H2O Frame → cuDF DataFrame

### H2O의 기법

H2O는 데이터를 **H2O Frame**이라는 분산 인메모리 데이터프레임으로 관리한다.  
데이터를 한번 로드하면 모델 훈련, 평가, 예측까지 메모리에서 나가지 않는 구조이다.

```python
# H2O 방식 (CPU, Java 기반)
train = h2o.import_file("train.csv")  # → JVM 힙 메모리에 로드
```

### GPU 적용 방향

cuDF DataFrame으로 데이터를 **GPU VRAM에 직접 로드**하고, 이후 전처리 → 훈련 → CV → 앙상블까지 데이터가 GPU를 떠나지 않게 구성한다.

```python
# GPU 적용 방향
import cudf
train = cudf.read_csv("train.csv")    # → GPU VRAM에 직접 로드
```

### 왜 이게 중요한가: CPU↔GPU 전송 오버헤드

H2O에서 XGBoost GPU 모드를 사용하면 다음과 같은 경로로 데이터가 이동한다:

```
H2O XGBoost GPU 모드:
  H2O Frame (JVM 힙) → Java-C++ 브릿지 → XGBoost C++ → GPU 전송 → 훈련
  
  이 브릿지 오버헤드 때문에 native XGBoost 대비 8~11배 느림
  (출처: szilard/GBM-perf 벤치마크, V100 GPU)
```

cuDF 기반이면 이 문제 자체가 없다:

```
cuDF 기반 파이프라인:
  cuDF DataFrame (GPU VRAM) → cuML/XGBoost GPU → 훈련
  
  데이터가 처음부터 끝까지 GPU에 상주
```

| 구분 | H2O Frame | cuDF DataFrame |
|------|-----------|----------------|
| 메모리 위치 | JVM 힙 (CPU) | GPU VRAM |
| 데이터 포맷 | 컬럼 기반 (자체 구현) | Apache Arrow 기반 컬럼형 |
| GPU 모델 연동 | Java-C++ 브릿지 필요 | Zero-copy 직접 전달 |
| 전처리 가속 | CPU 멀티스레드 | GPU 병렬 연산 |
| pandas 호환성 | `.as_data_frame()` 변환 필요 | `cudf.DataFrame` ↔ `pandas.DataFrame` 직접 변환 |

---

## 2. Algorithm Pool: H2O 알고리즘 → cuML/XGBoost GPU 1:1 매핑

### H2O의 기법

H2O AutoML은 다양한 알고리즘 family를 의도적으로 섞어서 **model diversity**를 확보한다.  
XGBoost, GBM, RF, GLM, DNN, XRT를 모두 돌리는 것이 핵심 전략이다.

### GPU 1:1 매핑 표

| H2O 알고리즘 (CPU/Java) | GPU 네이티브 대체 | 라이브러리 | 비고 |
|---|---|---|---|
| H2O GBM | XGBoost (`tree_method='hist', device='cuda'`) | XGBoost | GPU 가속 완전 지원 |
| XGBoost (Java wrapper) | XGBoost native GPU | XGBoost | Java 브릿지 제거, 직접 사용 |
| Random Forest (DRF) | `cuml.ensemble.RandomForestClassifier` | cuML | GPU 네이티브 RF |
| GLM | `cuml.linear_model.LogisticRegression` | cuML | GPU 가속 선형 모델 |
| Deep Neural Net | `torch.nn.Sequential` (MLP) | PyTorch + cuDNN | GPU 네이티브 DNN |
| XRT (Extremely Randomized Trees) | cuML RF (`max_features` 조정으로 다양성 확보) | cuML | 완전한 XRT 모방은 불가, 유사 효과 |

> **출처**: [cuML sklearn 호환 API](https://docs.rapids.ai/api/cuml/stable/cuml_intro/), [XGBoost GPU Training](https://xgboost.readthedocs.io/en/latest/python/dask-examples/gpu_training.html)
> **참고**: `tree_method='gpu_hist'`는 deprecated. 현재 올바른 설정: `tree_method='hist', device='cuda'`

### 핵심 차이

```
H2O AutoML:     XGBoost만 GPU 가속 가능, 나머지는 전부 CPU
GPU 재설계:     파이프라인의 모든 알고리즘이 GPU에서 실행
```

H2O의 철학이 "빠른 인프라로 단순한 서칭을 많이 돌리자"인데,  
**전체 알고리즘이 GPU 가속**되면 이 철학이 비로소 완전히 실현된다.

cuML은 scikit-learn 호환 API를 제공하므로, 기존 sklearn 기반 코드를  
`from sklearn` → `from cuml`로 import만 바꿔도 GPU 가속이 적용된다.

---

## 3. Training Pipeline: 순서 전략 계승 + GPU 병렬화 확장

### H2O의 기법

H2O AutoML은 의도적인 훈련 순서를 설계했다:

1. **Pre-specified 모델** → 안정적인 기본 파라미터로 baseline 확보
2. **Model diversity 확장** → 다양한 알고리즘 family 추가
3. **Random Search** → 하이퍼파라미터 탐색 범위 확대
4. **Adaptive Extension** → 시간이 남으면 상위 그리드 재시작

모든 모델은 **순차적으로 하나씩** 훈련된다.

### GPU에서의 확장

순서 전략 자체는 GPU에서도 유효하다. 다만 GPU 환경에서는 한 단계 더 나아갈 수 있다.

| 단계 | H2O (CPU) | GPU 적용 방향 |
|------|-----------|---------------|
| Phase 1: Baseline | Pre-specified 파라미터로 순차 훈련 (수 분) | 동일 전략, GPU로 **수 초~수십 초**에 완료 |
| Phase 2: Diversity | 모델 하나씩 순차 추가 | **여러 하이퍼파라미터 조합을 동시에** GPU에서 훈련 |
| Phase 3: Random Search | 하나씩 순차 탐색 | Dask/CUDA Stream으로 **병렬 하이퍼파라미터 탐색** |
| Phase 4: Adaptive | 남는 시간에 추가 탐색 | GPU로 기본 탐색이 빨리 끝나 **"남는 시간"이 훨씬 많음** |

### 병렬 탐색의 원리

```
H2O (CPU, 순차 실행):
  params_1 → train → eval → params_2 → train → eval → params_3 → ...
  총 시간: N × T

GPU 병렬 실행 (Dask + multi-GPU):
  GPU0: params_1 → train → eval
  GPU1: params_2 → train → eval     동시 실행
  GPU0: params_3 → train → eval
  총 시간: ~N/G × T  (G = GPU 수)
```

GPU VRAM이 충분하면 같은 GPU에서도 여러 모델을 동시에 훈련할 수 있고,  
Dask-CUDA의 `LocalCUDACluster`를 활용하면 멀티 GPU 병렬화도 가능하다.

---

## 4. Cross-Validation & Level-One Data: GPU에서 가장 큰 가속을 얻는 구간

### H2O의 기법

모든 base model을 동일한 k-fold로 cross-validation하고,  
**Out-of-Fold(OOF) prediction**을 모아서 N × L **level-one data**를 생성한다.

이것이 Stacked Ensemble의 입력이 된다.

### 왜 이 구간이 가장 느린가

5-fold CV × L개 모델 = **총 5L번 훈련**  
예를 들어 모델이 20개이면 총 100번 훈련해야 한다.  
AutoML에서 시간을 가장 많이 잡아먹는 구간이며,  
동시에 GPU 가속이 가장 직접적으로 효과를 주는 구간이다.

### GPU 적용 방향: Fold-Level Parallelism

H2O는 fold를 순차적으로 처리하지만, 멀티 GPU 환경에서는 **fold별 병렬 처리**가 가능하다.

```
H2O (CPU, 순차):
  모델A-Fold1 → 모델A-Fold2 → 모델A-Fold3 → 모델A-Fold4 → 모델A-Fold5
  총 시간: 5T

GPU 재설계 (멀티 GPU 병렬):
  GPU0: 모델A-Fold1 ──→
  GPU1: 모델A-Fold2 ──→
  GPU0: 모델A-Fold3 ──→       총 시간: ~T (이론적 5배 가속)
  GPU1: 모델A-Fold4 ──→
  GPU0: 모델A-Fold5 ──→
```

### Level-One Data 생성도 GPU에서

| 단계 | H2O (CPU) | GPU 적용 방향 |
|------|-----------|---------------|
| OOF 예측 생성 | CPU에서 예측 | GPU에서 직접 예측 (cuML predict) |
| 예측 결과 수집 | H2O Frame으로 병합 | cuDF DataFrame으로 GPU 메모리에서 concat |
| Meta learner 전달 | JVM 내부 전달 | **GPU 메모리를 떠나지 않고** 바로 다음 단계로 전달 |

cuML 기반 Stacking에서 10만 행, 10 feature 데이터 기준  
**훈련 35배, 추론 350배 이상 빨라진 벤치마크**가 보고되었으며,  
이는 CV 내부가 GPU에서 돌아가기 때문에 가능한 수치이다.

---

## 5. Stacked Ensemble: H2O 핵심 기법의 GPU 강화 방향

### H2O의 기법 (요약)

- **Two-type ensemble**: All Models (최고 성능) + Best of Family (프로덕션 배포용)
- **Meta learner**: Non-negative GLM with Lasso/Elastic Net → sparse ensemble 유도
- **OOF 기반**: Cross-validation으로 데이터 누수 방지

### GPU 적용 방향

#### 기법 1: Two-type Ensemble 구조 — 그대로 계승

이것은 코드가 아니라 **설계 전략**이다.  
"전체 모델 스태킹"과 "family별 best 모델 스태킹" 두 가지를 모두 만드는 전략은  
GPU든 CPU든 동일하게 적용 가능하다.

| 앙상블 유형 | 구성 | 목적 | GPU 적용 |
|---|---|---|---|
| All Models | 모든 base model 포함 | 최고 성능 추구 | 동일 구조 |
| Best of Family | 알고리즘별 최고 1개씩 | 속도 + 안정성 | 동일 구조 (GPU 메모리 절약에도 유리) |

#### 기법 2: Meta Learner — cuML GLM으로 대체

H2O의 meta learner인 non-negative GLM을 cuML로 대체한다.

| 구분 | H2O | GPU 적용 |
|------|-----|----------|
| 구현 | H2O GLM (Java) | `cuml.linear_model.LogisticRegression` / `LinearRegression` |
| 실행 위치 | CPU | GPU |
| 입력 데이터 이동 | Level-one data가 JVM 내부에서 전달 | Level-one data가 GPU VRAM에 이미 있으므로 **이동 zero** |
| Regularization | L1 (Lasso) / Elastic Net | cuML에서 `penalty='elasticnet'` 동일 지원 |

#### 기법 3: GPU에서만 가능한 확장 — 더 복잡한 Meta Learner

H2O가 GLM을 meta learner로 쓴 이유 중 하나는 CPU에서 빨리 돌아야 해서이다.  
GPU 기반이면 **meta learner로 XGBoost나 가벼운 MLP**를 써볼 여유가 생긴다.

H2O에서도 `metalearner_algorithm` 파라미터로 지원은 하지만,  
CPU 기반에서는 실험 비용이 높아 실무에서 잘 쓰이지 않았다.  
GPU에서는 이 실험 비용이 극적으로 줄어든다.

#### GPU 메모리 관리 전략

Stacking 시 생성되는 level-one data 크기: **샘플 수(N) × 모델 수(L)**

예: 1,000,000 샘플 × 100 모델 → 매우 큰 행렬 → GPU OOM 가능

| 해결 전략 | 설명 |
|-----------|------|
| Best of Family 활용 | 모델 수를 5~6개로 제한 → 메모리 부담 대폭 감소 |
| 스트리밍 처리 | 전체 데이터를 한 번에 처리하지 않고 배치 단위로 OOF 생성 |
| GPU 메모리 상주 (Zero-copy) | CPU↔GPU 이동 최소화 → cuDF DataFrame 위에서 직접 처리 |
| Dask-cuDF 분산 처리 | 데이터가 단일 GPU VRAM을 초과할 경우 멀티 GPU 분산 |

---

## 6. Leaderboard & Explainability

### H2O의 기법

- Leaderboard에 성능 메트릭 + `training_time_ms` + `predict_time_per_row_ms` 포함
- `explain()` 한 줄로 SHAP, PDP, Variable Importance, Model Correlation 생성

### GPU 적용 방향

이 시스템은 **모델 학습 레이어 위에 있는 분석/시각화 레이어**이므로  
GPU/CPU와 독립적으로 거의 그대로 이식 가능하다.

다만 GPU 환경에서 추가로 가능한 것:

| 항목 | H2O (CPU) | GPU 확장 |
|------|-----------|----------|
| SHAP 계산 | TreeSHAP (CPU) | **GPUTreeSHAP** (NVIDIA) — 대규모 데이터 SHAP 대폭 가속 |
| 예측 속도 측정 | CPU 추론 기준 | GPU 추론 기준 — 프로덕션 배포 시 더 현실적인 지표 |
| Leaderboard 구성 | H2O 내장 | pandas/cuDF DataFrame으로 자유롭게 커스텀 가능 |

---

## 7. 전체 아키텍처: GPU AutoML 파이프라인

```
┌──────────────────────────────────────────────────────────────┐
│                    GPU Data Layer                            │
│  cuDF DataFrame으로 GPU VRAM에 데이터 적재 및 상주            │
└──────────────────────┬───────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────────────────────┐
│               Feature Engineering                            │
│  cuDF 기반 GPU 전처리 + CV fold 분할 준비                     │
└──────────────────────┬───────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────────────────────┐
│            Phase 1: Quick Baseline                           │
│  Pre-specified 파라미터로 XGBoost GPU + cuML RF + cuML GLM    │
│  → 수 초~수십 초 안에 baseline 확보                           │
└──────────────────────┬───────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────────────────────┐
│         Phase 2-3: HPO + Parallel Training                   │
│  Random Search + Dask-CUDA 병렬 하이퍼파라미터 탐색            │
│  XGBoost / cuML RF / cuML GLM / PyTorch MLP 동시 훈련        │
│  + k-fold CV (fold-level parallelism)                        │
│  → OOF prediction 생성 (GPU 메모리에서)                       │
└──────────────────────┬───────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────────────────────┐
│              Stacked Ensemble                                │
│  ┌─────────────────┐  ┌──────────────────────┐              │
│  │  All Models      │  │  Best of Family       │              │
│  │  (최고 성능)      │  │  (속도 + 메모리 절약)  │              │
│  └────────┬────────┘  └───────────┬──────────┘              │
│           │                       │                          │
│           ▼                       ▼                          │
│     cuML GLM Meta Learner (GPU에서 직접 학습)                 │
└──────────────────────┬───────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────────────────────┐
│            Leaderboard + Explainability                      │
│  성능 순위표 + GPUTreeSHAP + Variable Importance              │
│  → 최종 배포 모델 선택 (Best Accuracy / Best Latency)         │
└──────────────────────────────────────────────────────────────┘
```

---

## 8. H2O 기법 → GPU 적용 매핑 요약

| H2O 기법 | GPU 적용 방향 | 사용 라이브러리 | 난이도 |
|----------|--------------|----------------|--------|
| H2O Frame (인메모리) | cuDF DataFrame (GPU VRAM 상주) | cuDF | 낮음 |
| XGBoost (Java wrapper) | XGBoost native GPU | XGBoost | 낮음 |
| GBM | XGBoost `hist` + `device='cuda'` | XGBoost | 낮음 |
| Random Forest | cuML RandomForest | cuML | 낮음 |
| GLM | cuML LogisticRegression/LinearRegression | cuML | 낮음 |
| DNN | PyTorch MLP + cuDNN | PyTorch | 중간 |
| XRT | cuML RF (`max_features` 조정) | cuML | 중간 (완전 모방 불가) |
| 훈련 순서 전략 | 동일 순서 + GPU 병렬 탐색 | Dask-CUDA | 중간 |
| k-fold CV + OOF | GPU 가속 CV + fold-level parallelism | cuML + Dask | 중간 |
| Stacked Ensemble (2종) | 동일 구조 + cuML GLM meta learner | cuML | 낮음 |
| Random Search | 동일 + 병렬 trial 실행 | Dask-CUDA | 중간 |
| Leaderboard | pandas/cuDF DataFrame으로 구성 | pandas/cuDF | 낮음 |
| Explainability (SHAP) | GPUTreeSHAP | SHAP + NVIDIA | 중간 |

> **핵심**: 대부분의 기법이 "낮음" 난이도로 적용 가능하다.  
> 이는 cuML이 scikit-learn 호환 API를 제공하고, XGBoost는 파라미터만 바꾸면 되기 때문이다.  
> 커널 수준의 재작성이 필요한 것은 하나도 없다.
>
> **출처**: [cuML 공식 문서](https://docs.rapids.ai/api/cuml/stable/cuml_intro/), [Dask-CUDA API](https://docs.rapids.ai/api/dask-cuda/nightly/api/), [XGBoost GPU Docs](https://xgboost.readthedocs.io/en/latest/python/dask-examples/gpu_training.html)

---

## 9. 기대 효과: 벤치마크 사례

### 사례 1: TPOT + RAPIDS

TPOT(AutoML 프레임워크)이 RAPIDS cuML과 XGBoost GPU를 통합한 결과:

- GPU 가속 AutoML이 **1시간** 만에 CPU **8시간**보다 높은 정확도의 파이프라인 발견
- 최종 정확도: CPU 대비 **+2%, +1.3%** 향상
- 원리: GPU가 빨라서 같은 시간에 더 많은 탐색 → 더 좋은 모델 발견 확률 증가

> 출처: [Faster AutoML with TPOT and RAPIDS](https://medium.com/rapids-ai/faster-automl-with-tpot-and-rapids-758455cd89e5)

### 사례 2: AutoGluon + RAPIDS

- 훈련 속도: 최대 **40배** 빠름
- 추론 속도: 최대 **10배** 빠름

> 출처: [NVIDIA Blog: Advancing AutoML 10x Faster](https://developer.nvidia.com/blog/advancing-the-state-of-the-art-in-automl-now-10x-faster-with-nvidia-gpus-and-rapids/)

### 사례 3: cuML StackingClassifier 벤치마크

- 10만 행, 10 feature 데이터 기준
- 훈련: **35배** 가속
- 추론: **350배** 이상 가속

> 출처: [100x Faster ML Ensembling with RAPIDS cuML](https://medium.com/rapids-ai/100x-faster-machine-learning-model-ensembling-with-rapids-cuml-and-scikit-learn-meta-estimators-d869788ee6b1)

### 의미

이 벤치마크들은 H2O의 핵심 철학인  
**"빠른 인프라로 단순한 서칭을 많이 돌려서 좋은 모델을 찾자"**를  
GPU 환경에서 극단적으로 실현할 수 있음을 보여준다.

- CPU에서 1시간에 20개 모델 → GPU에서 1시간에 200개 이상 모델 탐색 가능
- 더 많은 탐색 = 더 좋은 모델을 찾을 확률 증가
- Stacked Ensemble의 base model이 많아질수록 앙상블 성능도 향상

---

## 정리

| 관점 | 결론 |
|------|------|
| H2O 코드 재사용 | 불가 (Java/CPU 종속) |
| H2O 기법 재사용 | **가능** — 설계 전략은 GPU에서도 유효 |
| 커널 재작성 필요성 | **없음** — cuML, XGBoost GPU, cuDF 등 기존 라이브러리 조합으로 충분 |
| 가장 큰 가속 구간 | Cross-Validation + Level-One Data 생성 (5L번 훈련 → fold-level 병렬화) |
| GPU에서만 가능한 확장 | 전체 알고리즘 GPU 가속, 병렬 HPO, 복잡한 meta learner 실험 |
| 실용적 난이도 | 대부분 **낮음** (API 호환, 파라미터 변경 수준) |

> **"H2O의 기법은 가져가되, 구현은 GPU 네이티브로 새로 짠다"**  
> 이 접근이 가능한 이유는 RAPIDS 생태계(cuDF, cuML)가  
> H2O가 커버하는 알고리즘 풀을 GPU 네이티브로 거의 다 대체할 수 있기 때문이다.